In [2]:
import pandas as pd
import requests
import os
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path

In [3]:
load_dotenv(override=True)

API_KEY = os.getenv("OPEN_WEATHER_MAP_API_KEY")
LAT = float(os.getenv("LATITUDE"))
LON = float(os.getenv("LONGITUDE"))

In [5]:
def get_historical_weather(ts):
    unix_ts = int(ts.timestamp())
    url = f"https://api.openweathermap.org/data/3.0/onecall/timemachine?lat={LAT}&lon={LON}&dt={unix_ts}&appid={API_KEY}"
    response = requests.get(url).json()

    if "data" in response:
        weather = response["data"][0]
        return pd.Series({
            "timestamp": weather.get('dt'),
            "wind_speed": weather.get('wind_speed'),
            "wind_deg": weather.get('wind_deg'),
            "pressure": weather.get('pressure'),
            
        })

In [6]:
data = pd.read_parquet(Path("../data/data.parquet"))
data['timestamp'] = pd.to_datetime(data["period.datetime_from.utc"])
ts_sample = data.loc[:5, "timestamp"].dt.round('h')
unique_hours = pd.Series(ts_sample.unique())
unix_start = int(unique_hours[0].timestamp())
unix_end = int(unique_hours[5].timestamp())

In [7]:
url = "https://api.openweathermap.org/data/3.0/onecall/timemachine?lat={lat}&lon={lon}&dt={time}&appid={api_key}"
tx = ts_sample[0]
response = requests.get(url.format(lat=LAT, lon=LON, time=int(tx.timestamp()), api_key=API_KEY)).json()
response

{'lat': 48.1373,
 'lon': 11.5649,
 'timezone': 'Europe/Berlin',
 'timezone_offset': 3600,
 'data': [{'dt': 1735686000,
   'sunrise': 1735715047,
   'sunset': 1735745414,
   'temp': 272.54,
   'feels_like': 272.54,
   'pressure': 1029,
   'humidity': 79,
   'dew_point': 269.72,
   'clouds': 0,
   'wind_speed': 0.45,
   'wind_deg': 280,
   'wind_gust': 1.34,
   'weather': [{'id': 800,
     'main': 'Clear',
     'description': 'clear sky',
     'icon': '01n'}]}]}

In [10]:
response["data"]

[{'dt': 1735686000,
  'sunrise': 1735715047,
  'sunset': 1735745414,
  'temp': 272.54,
  'feels_like': 272.54,
  'pressure': 1029,
  'humidity': 79,
  'dew_point': 269.72,
  'clouds': 0,
  'wind_speed': 0.45,
  'wind_deg': 280,
  'wind_gust': 1.34,
  'weather': [{'id': 800,
    'main': 'Clear',
    'description': 'clear sky',
    'icon': '01n'}]}]

In [42]:
pd.DataFrame([response["data"][0], response["data"][0]])

,dt,sunrise,sunset,temp,feels_like,pressure,humidity,dew_point,clouds,wind_speed,wind_deg,wind_gust,weather
0,1735686000,1735715047,1735745414,272.54,272.54,1029,79,269.72,0,0.45,280,1.34,"[{'id': 800, 'main': 'Clear', 'description': '..."
1,1735686000,1735715047,1735745414,272.54,272.54,1029,79,269.72,0,0.45,280,1.34,"[{'id': 800, 'main': 'Clear', 'description': '..."


In [24]:
out = get_historical_weather(ts_sample[0])

In [25]:
out

wind_speed       0.45
wind_deg       280.00
pressure      1029.00
dtype: float64

In [43]:
ts_sample

0   2024-12-31 23:00:00+00:00
1   2025-01-01 00:00:00+00:00
2   2025-01-01 01:00:00+00:00
3   2025-01-01 02:00:00+00:00
4   2025-01-01 03:00:00+00:00
5   2025-01-01 04:00:00+00:00
Name: timestamp, dtype: datetime64[ns, UTC]

In [11]:
dt_range = pd.date_range(
    start=pd.to_datetime("1/1/2018").tz_localize("Europe/Berlin"),
    end=pd.to_datetime("1/1/2018").tz_localize("Europe/Berlin"), freq="1h")

In [ ]:
pd.to_datetime("2025-01-01 00:00:00+01:00")

Timestamp('2024-12-31 23:00:00+0000', tz='UTC')

In [2]:
from datetime import datetime
datetime(2026,1,1)

datetime.datetime(2026, 1, 1, 0, 0)

In [92]:
dt_range[0].timestamp()

1514761200.0

In [ ]:
dt = dt_range[0]
print(dt)
ts = datetime.timestamp(dt)
print(ts)
pd.to_datetime(datetime.fromtimestamp(ts, tz=dt.tzinfo))

In [162]:
url = "https://api.openweathermap.org/data/3.0/onecall/timemachine?lat={lat}&lon={lon}&dt={time}&appid={api_key}&units=metric"
records = []
unix_ts = int(datetime.timestamp(dt_range[0]))
response = requests.get(url.format(lat=LAT, lon=LON, time=unix_ts, api_key=API_KEY)).json()
if "data" in response:
    records.append(response["data"][0])
else:
    print("No data.")

In [163]:
df = pd.DataFrame(records)

In [164]:
df.head()

,dt,sunrise,sunset,temp,feels_like,pressure,humidity,dew_point,clouds,wind_speed,wind_deg,weather
0,1514761200,1514790247,1514820602,2.94,-0.41,1008,86,0.83,3,3.6,207,"[{'id': 800, 'main': 'Clear', 'description': '..."


In [165]:
df["ts"] = df["dt"].apply(lambda x: datetime.fromtimestamp(x, tz=dt.tzinfo))

In [166]:
df.head()

,dt,sunrise,sunset,temp,feels_like,pressure,humidity,dew_point,clouds,wind_speed,wind_deg,weather,ts
0,1514761200,1514790247,1514820602,2.94,-0.41,1008,86,0.83,3,3.6,207,"[{'id': 800, 'main': 'Clear', 'description': '...",2018-01-01 00:00:00+01:00


In [98]:
pd.to_datetime(1645888976, origin="unix")

Timestamp('1970-01-01 00:00:01.645888976')

In [97]:
pd.to_datetime(1735686000, origin="unix")

Timestamp('1970-01-01 00:00:01.735686')

In [106]:
pd.to_datetime(datetime.fromtimestamp(1514761200.0))

Timestamp('2018-01-01 00:00:00')

In [105]:
datetime.timestamp(dt_range[0])

1514761200.0

In [119]:
dt = dt_range[0]

In [149]:
dt.tzinfo

<DstTzInfo 'Europe/Berlin' CET+1:00:00 STD>

In [ ]:
pd.to_datetime(datetime.fromtimestamp())

In [ ]:
from datetime import tzinfo

In [144]:
dt = dt_range[0]
print(dt)

conv = (dt - pd.Timestamp("1970-01-01 00:00:00+0100")) // pd.Timedelta("1s")
pd.to_datetime(datetime.fromtimestamp(conv))

2018-01-01 00:00:00+01:00


Timestamp('2018-01-01 01:00:00')

In [155]:
dt = dt_range[0]
print(dt)
ts = datetime.timestamp(dt)
print(ts)
pd.to_datetime(datetime.fromtimestamp(ts, tz=dt.tzinfo))

2018-01-01 00:00:00+01:00
1514761200.0


Timestamp('2018-01-01 00:00:00+0100', tz='Europe/Berlin')

In [159]:
url = "https://api.openweathermap.org/data/3.0/onecall/day_summary?lat={lat}&lon={lon}&date={date}&appid={api_key}"

In [160]:
js = requests.get(url.format(
    lat=LAT,
    lon=LON,
    date="2025-01-01",
    api_key=API_KEY
)).json()

In [161]:
js

{'lat': 48.13725200011463,
 'lon': 11.564923999614544,
 'tz': '+01:00',
 'date': '2025-01-01',
 'units': 'standard',
 'cloud_cover': {'afternoon': 72.0},
 'humidity': {'afternoon': 49.0},
 'precipitation': {'total': 0.0},
 'temperature': {'min': 269.66,
  'max': 281.95,
  'afternoon': 281.37,
  'night': 271.94,
  'evening': 276.1,
  'morning': 270.69},
 'pressure': {'afternoon': 1026.0},
 'wind': {'max': {'speed': 4.91, 'direction': 229.0}}}